In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import numpy as np
import time
import os
from pathlib import Path
import json
import math

In [2]:
df = pd.read_csv('anime_df.csv')
df.columns.values[0] = 'usernames'

In [3]:
df = df.set_index(df.columns[0])
df = df.astype(float)

In [4]:
conf_path = Path("../tauri.conf.json")

with open(conf_path, 'r') as f:
    config = json.load(f)

In [5]:
app_data_path = Path(os.getenv('APPDATA') or Path.home() / ".local/share")

# Path objects handle joining naturally with the / operator
watchlist_path = app_data_path / config['identifier'] / 'watchlist.json'
notwatchlist_path = app_data_path / config['identifier'] / 'notwatchlist.json'

print(f"Watchlist path: {watchlist_path}")
print(f"Not atchlist path: {notwatchlist_path}")
print(f"DEBUG: Próbuję czytać z: {os.path.abspath(watchlist_path)}")

Watchlist path: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\watchlist.json
Not atchlist path: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\notwatchlist.json
DEBUG: Próbuję czytać z: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\watchlist.json


In [6]:
user_data = pd.read_json(watchlist_path)
not_wanted = pd.read_json(notwatchlist_path)


# Konwersja na listę słowników
user_data = user_data.to_dict(orient='records')
not_wanted = not_wanted.to_dict(orient='records')
not_wanted_set = {int(elem['mal_id']) for elem in not_wanted if pd.notna(elem['mal_id'])}
print(not_wanted_set)

{48897, 21507, 56835, 27525, 137, 908, 12685, 37264, 36497, 62352, 38419, 37781, 37402, 54042, 20509, 61345, 18851, 38693, 49703, 35118, 45999, 42030, 34100, 44983, 8888, 59833, 59968, 62149, 51781, 26057, 59465, 36683, 16468, 53588, 44248, 31706, 10842, 33372, 15197, 61149, 37087, 51168, 36833, 49633, 8171, 14829, 61293, 17277}


In [ ]:
def get_recommendations(user, df_orig, df_norm, n=5, k=10):
    knn = NearestNeighbors(n_neighbors=k+1, metric='cosine')
    knn.fit(df_norm)
    
    user_idx = df_norm.index.get_loc(user)
    distances, indices = knn.kneighbors(df_norm.iloc[[user_idx]])
    
    similarities = 1 - distances.flatten()
    neighbor_indices = indices.flatten()
    

    similar_users = pd.Series(similarities[1:], index=df_norm.index[neighbor_indices[1:]])
    
    user_ratings = df_orig.loc[user]
    user_mean = user_ratings[user_ratings != 0].mean() if (user_ratings != 0).any() else 0
    

    neighbor_ratings = df_orig.loc[similar_users.index]
    candidate_mask = (neighbor_ratings > 0).any(axis=0) & (user_ratings == 0)
    candidate_anime = neighbor_ratings.columns[candidate_mask]
    
    predictions = {}
    for anime in candidate_anime:
        id_ref, name = anime.split('_', 1)
        if(int(id_ref)) in not_wanted_set:
            continue        
        
        relevant_indices = similar_users.index[neighbor_ratings[anime] > 0]
        
        if len(relevant_indices) > 0:
            weights = similar_users.loc[relevant_indices]
            norm_ratings = df_norm.loc[relevant_indices, anime]
            
            if weights.sum() > 0: pass
            pred_deviation = np.average(norm_ratings, weights=weights)
            predictions[anime] = user_mean + pred_deviation
    
    print("AAA", len(predictions))
    if not predictions:
        return {}
    recommendations = pd.Series(predictions).sort_values(ascending=False)
    return recommendations.head(n).to_dict()

In [ ]:
t0 = time.time()
print("BBB")

In [9]:
user_ratings = {
    #f"{entry['mal_id']}": f"{entry['score']}"
    f"{entry['mal_id']}_{entry['title']}": entry['score']
    for entry in user_data
    if entry.get('score') is not None and not (isinstance(entry['score'], float) and math.isnan(entry['score']))
}

print(user_ratings)

{'502_Dragon Ball Movie 1: Shen Long no Densetsu': 9.0, '891_Dragon Ball Movie 2: Majinjou no Nemurihime': 9.0, '892_Dragon Ball Movie 3: Makafushigi Daibouken': 9.0, '223_Dragon Ball': 10.0, '894_Dragon Ball Z Movie 01: Ora no Gohan wo Kaese!!': 10.0, '895_Dragon Ball Z Movie 02: Kono Yo de Ichiban Tsuyoi Yatsu': 10.0, '896_Dragon Ball Z Movie 03: Chikyuu Marugoto Choukessen': 10.0, '897_Dragon Ball Z Movie 04: Super Saiyajin da Son Gokuu': 10.0, '898_Dragon Ball Z Movie 05: Tobikkiri no Saikyou tai Saikyou': 10.0, '6714_Dragon Ball Z: Atsumare! Gokuu World': 10.0, '899_Dragon Ball Z Movie 06: Gekitotsu!! 100-oku Power no Senshi-tachi': 10.0, '900_Dragon Ball Z Movie 07: Kyokugen Battle!! Sandai Super Saiyajin': 10.0, '901_Dragon Ball Z Movie 08: Moetsukiro!! Nessen, Ressen, Chougekisen': 10.0, '902_Dragon Ball Z Movie 09: Ginga Girigiri!! Bucchigiri no Sugoi Yatsu': 10.0, '984_Dragon Ball Z: Saiya-jin Zetsumetsu Keikaku': 10.0, '903_Dragon Ball Z Movie 10: Kiken na Futari! Super Sens

In [10]:
new_user_name = "Current_User"

new_user_row = pd.Series(0, index=df.columns, name=new_user_name)

for anime, rating in user_ratings.items():
    if anime in new_user_row.index:
        new_user_row[anime] = rating


df_extended = pd.concat([df, new_user_row.to_frame().T])
df_normalized = df_extended.apply(lambda row: row - row[row != 0].mean() if (row != 0).any() else row, axis=1)

In [11]:
recommendations = get_recommendations('Current_User', df_extended, df_normalized, n=5, k=5)

for anime_key, score in recommendations.items():
    id_ref, name = anime_key.split('_', 1)
    print(f"Recommend: {name} (ID: {id_ref}) with predicted score: {score:.2f}")

Recommend: Naruto: Shippuuden (ID: 1735) with predicted score: 10.53
Recommend: Kiseijuu: Sei no Kakuritsu (ID: 22535) with predicted score: 10.53
Recommend: Gyakuten Saiban: Sono "Shinjitsu", Igi Ari! (ID: 31630) with predicted score: 10.46
Recommend: Koe no Katachi (ID: 28851) with predicted score: 10.46
Recommend: Gyakuten Saiban: Sono "Shinjitsu", Igi Ari! Season 2 (ID: 37490) with predicted score: 10.46


In [12]:
print(time.time() - t0)

0.9750738143920898
